In [8]:
! uv pip install langchain groq tiktoken rapidocr-onnxruntime python-dotenv langchain-community 


Using Python 3.13.15 environment at: c:\Users\windows\Desktop\projects\llmops_series\.venv
Checked 6 packages in 21ms


In [5]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

## data ingestion 

In [9]:
from langchain_community.document_loaders import TextLoader

In [14]:
loader=TextLoader('/Users/windows/Desktop/projects/llmops_series/data/Agentic Ai.txt',encoding="utf8")
documents=loader.load()

In [15]:
documents

[Document(metadata={'source': '/Users/windows/Desktop/projects/llmops_series/data/Agentic Ai.txt'}, page_content='Agentic AI\n\nAgentic AI refers to artificial intelligence systems that can autonomously plan, reason, make decisions, and take actions to achieve a specific goal. Unlike traditional chatbots that mainly generate a response to a user\'s question, AI agents can interact with external tools, retrieve information, execute actions, evaluate results, and continue working until a task is completed.\n\nAn AI agent typically consists of several components, including a large language model, tools, memory, planning capabilities, and an execution loop. The language model acts as the reasoning engine of the agent. Tools allow the agent to interact with external systems such as databases, APIs, search engines, calculators, file systems, and business applications.\n\nThe basic workflow of an AI agent can be described as understanding the user\'s goal, analyzing the available information,

In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [17]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=200,chunk_overlap=20)

In [18]:
text_chunks=text_splitter.split_documents(documents)

In [19]:
text_chunks

[Document(metadata={'source': '/Users/windows/Desktop/projects/llmops_series/data/Agentic Ai.txt'}, page_content='Agentic AI'),
 Document(metadata={'source': '/Users/windows/Desktop/projects/llmops_series/data/Agentic Ai.txt'}, page_content='Agentic AI refers to artificial intelligence systems that can autonomously plan, reason, make decisions, and take actions to achieve a specific goal. Unlike traditional chatbots that mainly generate'),
 Document(metadata={'source': '/Users/windows/Desktop/projects/llmops_series/data/Agentic Ai.txt'}, page_content="mainly generate a response to a user's question, AI agents can interact with external tools, retrieve information, execute actions, evaluate results, and continue working until a task is completed."),
 Document(metadata={'source': '/Users/windows/Desktop/projects/llmops_series/data/Agentic Ai.txt'}, page_content='An AI agent typically consists of several components, including a large language model, tools, memory, planning capabilities, a

In [21]:
! uv pip install faiss-cpu
! uv add langchain-huggingface sentence-transformers

Using Python 3.13.15 environment at: c:\Users\windows\Desktop\projects\llmops_series\.venv
Checked 1 package in 9ms
Resolved 104 packages in 1.77s
   Building llmops-series @ file:///C:/Users/windows/Desktop/projects/llmops_series
      Built llmops-series @ file:///C:/Users/windows/Desktop/projects/llmops_series
 Downloaded networkx
 Downloaded scikit-learn
 Downloaded scipy
 Downloaded torch
Prepared 12 packages in 32.29s
Uninstalled 1 package in 53ms
Installed 30 packages in 15.14s
 + annotated-doc==0.0.5
 + click==8.5.0
 + cloudpickle==3.1.2
 + filelock==4.0.1
 + fsspec==2026.9.0
 + hf-xet==1.6.0
 + huggingface-hub==1.32.0
 + jinja2==3.1.6
 + joblib==1.6.0
 + langchain-huggingface==1.2.2
 ~ llmops-series==0.1.0 (from file:///C:/Users/windows/Desktop/projects/llmops_series)
 + markdown-it-py==4.2.0
 + markupsafe==3.0.3
 + mdurl==0.1.2
 + mpmath==1.3.0
 + narwhals==2.26.0
 + networkx==3.7
 + rich==15.0.0
 + safetensors==0.8.0
 + scikit-learn==1.9.1
 + scipy==1.18.1
 + sentence-transf

In [22]:
#embedding model 
from langchain_huggingface import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

c:\Users\windows\Desktop\projects\llmops_series\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8230.71it/s]


In [23]:
#creating the faiss vector store 
from langchain_community.vectorstores import FAISS
vector_store = FAISS.from_documents(
    documents=text_chunks,
    embedding=embedding_model
)

In [33]:
retriever=vector_store.as_retriever()

In [25]:
query="what is the key characteristic of agentic ai"
docs=vector_store.similarity_search(query,k=4)
for i,doc in enumerate(docs):
    print(f"document {i+1}")
    print(doc.page_content)
    print("-"*50)

document 1
Agentic AI
--------------------------------------------------
document 2
The Future of Agentic AI
--------------------------------------------------
document 3
Agentic AI refers to artificial intelligence systems that can autonomously plan, reason, make decisions, and take actions to achieve a specific goal. Unlike traditional chatbots that mainly generate
--------------------------------------------------
document 4
Evaluating AI agents is important because an agent may produce different results depending on the tools it selects, the sequence of actions it takes, and the information it retrieves.
--------------------------------------------------


In [26]:
from langchain_core.prompts import ChatPromptTemplate

template="""You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use ten sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:
"""

In [27]:
prompt=ChatPromptTemplate.from_template(template)

In [28]:
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks.\nUse the following pieces of retrieved context to answer the question.\nIf you don't know the answer, just say that you don't know.\nUse ten sentences maximum and keep the answer concise.\nQuestion: {question}\nContext: {context}\nAnswer:\n"), additional_kwargs={})])

In [32]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()
#covert the llm response   from different format to the plain python str format ad give it to us 

In [35]:
from langchain_groq import ChatGroq

In [36]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

In [37]:
#runnable in langchain is something that can recieve an input and give us an ouput 
#runnablepasthrough means jo input mujhe mila hai use change kiye bina aage pass kardo 
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [38]:
rag_chain.invoke("tll me about agentic ai ")

'Agentic AI refers to artificial‑intelligence systems that can **autonomously plan, reason, make decisions, and take actions** to achieve a specific goal.  \nUnlike traditional chatbots, which mainly generate text responses, agentic AI includes a **reasoning component** that lets it evaluate options, set sub‑goals, and execute tasks without constant human direction.  \nThese systems combine large‑language models with tools such as planners, memory modules, and external APIs to interact with their environment.  \nThey can **self‑direct** their behavior, adapting strategies as conditions change, which makes them suitable for complex, multi‑step problems.  \nAgentic AI is often built as a hierarchy: a high‑level goal is broken down into actionable steps, each executed by specialized modules.  \nBecause they can **learn from feedback** and update their internal models, they improve performance over time.  \nApplications include autonomous research assistants, robotic process automation, an